In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visualizations
%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

dataset_path = '/content/drive/MyDrive/Tech Scholarship Data Science Curriculum & Materials/notebooks-&-dataset/dataset'
BASE_URL = f"{dataset_path}/"
# Define files
files = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv"
}

print("Downloading and caching Olist Datasets for auditing...")

try:
    orders = pd.read_csv(f"{BASE_URL}{files['orders']}")
    order_items = pd.read_csv(f"{BASE_URL}{files['order_items']}")
    products = pd.read_csv(f"{BASE_URL}{files['products']}")
    reviews = pd.read_csv(f"{BASE_URL}{files['reviews']}")

    print("\nDatasets successfully loaded into Memory:")
    print(f"- Orders: {orders.shape[0]} rows, {orders.shape[1]} columns.")
    print(f"- Order Items: {order_items.shape[0]} rows, {order_items.shape[1]} columns.")
    print(f"- Products: {products.shape[0]} rows, {products.shape[1]} columns.")
    print(f"- Reviews: {reviews.shape[0]} rows, {reviews.shape[1]} columns.")
except FileNotFoundError as e:
    print(f"Error loading from specified Google Drive path: {e}")
    print("Attempting to load from a generic local Colab content directory (/content/)...")
    # New fallback: try a different local path, e.g., /content/
    LOCAL_COLAB_PATH = "/content/"
    orders = pd.read_csv(f"{LOCAL_COLAB_PATH}{files['orders']}")
    order_items = pd.read_csv(f"{LOCAL_COLAB_PATH}{files['order_items']}")
    products = pd.read_csv(f"{LOCAL_COLAB_PATH}{files['products']}")
    reviews = pd.read_csv(f"{LOCAL_COLAB_PATH}{files['reviews']}")
    print("\nFallback Datasets successfully loaded from /content/:")
    print(f"- Orders: {orders.shape[0]} rows, {orders.shape[1]} columns.")
    print(f"- Order Items: {order_items.shape[0]} rows, {order_items.shape[1]} columns.")
    print(f"- Products: {products.shape[0]} rows, {products.shape[1]} columns.")
    print(f"- Reviews: {reviews.shape[0]} rows, {reviews.shape[1]} columns.")
except Exception as e:
    print(f"An unexpected error occurred during data loading: {e}")
    print("Please ensure the dataset files are correctly placed in either the specified Google Drive path or /content/.")

# Session 9: Understanding Model Performance

— How to Evaluate, Select, and Trust Machine Learning Models

---

## Learning Objectives
By the end of this session, you will be able to:
1. **Select and justify** appropriate evaluation metrics for both classification and regression tasks based on the business context and error costs.
2. **Construct and interpret** a Confusion Matrix to calculate Precision, Recall, F1-Score, and Accuracy.
3. **Deconstruct regression performance** using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and R-squared ($R^2$).
4. **Differentiate** between a Loss Function (used during training) and an Evaluation Metric (used for assessment).
5. **Diagnose and mitigate** Overfitting and Underfitting using training vs. validation performance.
6. **Apply Cross-Validation** to assess the stability and generalisability of your models on the Brazilian Olist e-commerce dataset.

---

## Section 1: Introduction — A Model Can Be Wrong in Different Ways

In Session 8, we built our first machine learning model using Scikit-Learn. We fed it data, trained it, and watched it output predictions. But how do we know if those predictions are actually *good*?

In the real world, **a model can be 95% accurate and still be completely useless—or even dangerous.**

Imagine Olist wants to deploy a model to predict whether a transaction is fraudulent.
* $99\%$ of all transactions on the platform are legitimate.
* If we write a "dumb" model that simply predicts `Is Fraud = False` for every single transaction, what is its accuracy?
* It is **$99\%$ accurate**!
* However, this model catches **$0\%$** of actual fraud cases, allowing fraudsters to operate freely. This is why evaluating machine learning models requires a deeper framework than just measuring

### 🧠 Reflection Question
Before we dive into the code, think about this:
*If you were ordering an expensive smartphone on Olist, which error would frustrate you more?*
1. The system falsely predicts your delivery will be late (but it actually arrives on time).
2. The system falsely predicts your delivery will be on time (but it actually arrives a week late).

As we will see, **not all errors are created equal.**

---

## Section 2: Two Different Types of Prediction Problems

Before choosing how to measure success, we must always align our evaluation metrics with the machine learning task. The two fundamental supervised learning tasks require completely different metric families:

| Feature | Classification | Regression |
| :--- | :--- | :--- |
| **Target Variable Type** | Categorical (discrete classes, e.g., `0` or `1`, `Low` or `High`) | Continuous (numeric values, e.g., prices, days, temperature) |
| **Olist Example** | Predicting if an order will receive a low review score (Rating $\le 2$) | Predicting the delivery duration in days |
| **Core Evaluation Focus** | Counting correct vs. incorrect classifications | Measuring the distance between predicted and actual values |

---

## Section 3: Why Accuracy Is Not Enough

**Accuracy** is defined as the ratio of correct predictions to total predictions:

$$\text{Accuracy} = \frac{\text{Number of Correct Predictions}}{\text{Total Number of Predictions}}$$

### The Class Imbalance Trap
When our dataset contains a highly imbalanced target class, accuracy fails completely. Let's see this in action using a quick python simulation.

In [ ]:
import numpy as np
import pandas as pd

# Simulated imbalance: 950 happy customers (0), 50 frustrated customers (1)
actual_labels = np.array([0] * 950 + [1] * 50)

# A lazy baseline model that always predicts 0 (majority class)
pred_labels = np.zeros_like(actual_labels)

# Calculate basic accuracy
accuracy = np.mean(actual_labels == pred_labels)
print(f"Simulated Imbalance Setup:")
print(f"- Total actual positive cases (1): {sum(actual_labels == 1)}")
print(f"- Total actual negative cases (0): {sum(actual_labels == 0)}")
print(f"- Dummy Model Accuracy: {accuracy * 100:.1f}%")

Simulated Imbalance Setup:
- Total actual positive cases (1): 50
- Total actual negative cases (0): 950
- Dummy Model Accuracy: 95.0%


---

## Section 4: Understanding the Confusion Matrix

To see exactly *how* a classification model is failing or succeeding, we use a **Confusion Matrix**. This tabular layout cross-tabulates actual values against predicted values:

```
                    PREDICTED CLASS
                  Negative    Positive
ACTUAL   Negative    TN          FP
CLASS    Positive    FN          TP
```

* **True Positive (TP):** We predicted positive, and it is positive (e.g., We predicted a low review, and the customer indeed left a 1-star review).
* **True Negative (TN):** We predicted negative, and it is negative (e.g., We predicted a good review, and the customer left a 5-star review).
* **False Positive (FP) [Type I Error]:** We predicted positive, but it is actually negative (e.g., We predicted a low review, but the customer was actually happy. We spent unnecessary customer service resources on them).
* **False Negative (FN) [Type II Error]:** We predicted negative, but it is actually positive (e.g., We predicted a good review, but the customer left a 1-star review. We missed an opportunity to intervene and save a churn-risk account).

---

## Section 5: Precision, Recall, and F1-Score

To handle imbalanced data and evaluate specific errors, we define three key metrics:

### 1. Precision
* **The Question:** *"Of all the cases we predicted as positive, how many were actually positive?"*
* **Formula:**
$$\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}$$
* **When to prioritize:** When the cost of a **False Positive** is very high (e.g., Mark spam emails. If we block a critical work email as spam [FP], the user loses important business).

### 2. Recall (Sensitivity)
* **The Question:** *"Of all the actual positive cases, how many did our model manage to find?"*
* **Formula:**
$$\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}$$
* **When to prioritize:** When the cost of a **False Negative** is extremely high (e.g., Catching terminal illnesses or flagging severe safety defects. Missing a defective brake pad [FN] is catastrophic).

### 3. F1-Score
* **The Question:** *"How can we balance Precision and Recall with a single metric?"*
* **Formula (Harmonic Mean):**
$$\text{F1-Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$
* **Why Harmonic Mean?** Unlike an arithmetic average, the harmonic mean penalises extreme imbalances. A model with $100\%$ precision but $0\%$ recall will have an F1-Score of $0$, whereas its arithmetic mean would misleadingly be $50\%$.

---

## Demonstration A: Implementing a Classification Model on Olist

Let's apply these concepts directly to the Olist dataset! We will predict whether an order will receive a **low review score (1 or 2 stars)** using logistic regression. This will help Olist proactively reach out to unhappy customers.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report

# 1. Target Construction: Define low review score (1 or 2 stars) as our positive class (1)
reviews_df = reviews.copy()
reviews_df['is_low_score'] = (reviews_df['review_score'] <= 2).astype(int)





In [ ]:
# Let's inspect class distribution
print("Class distribution for Low Review Scores:")
print(reviews_df['is_low_score'].value_counts(normalize=True))

Class distribution for Low Review Scores:
is_low_score
0    0.85311
1    0.14689
Name: proportion, dtype: float64


In [ ]:
# 2. To avoid target leakage and build a realistic model, we'll merge with orders
# to get delivery time difference as a predictor feature.
orders_df = orders.copy()
orders_df['delivery_time_days'] = (
    pd.to_datetime(orders_df['order_delivered_customer_date']) -
    pd.to_datetime(orders_df['order_purchase_timestamp'])
).dt.days

# Keep only completed deliveries to prevent missing values from breaking our demo
merged = pd.merge(reviews_df, orders_df, on='order_id').dropna(subset=['delivery_time_days'])

# Features (X) and Target (y)
X = merged[['delivery_time_days']]
y = merged['is_low_score']

merged.head()


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,is_low_score,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_time_days
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59,0,41dcb106f807e993532d446263290104,delivered,2018-01-11 15:30:49,2018-01-11 15:47:59,2018-01-12 21:57:22,2018-01-17 18:42:41,2018-02-02 00:00:00,6.0
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13,0,8a2e7ef9053dea531e4dc76bd6d853e6,delivered,2018-02-28 12:25:19,2018-02-28 12:48:39,2018-03-02 19:08:15,2018-03-09 23:17:20,2018-03-14 00:00:00,9.0
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24,0,e226dfed6544df5b7b87a48208690feb,delivered,2018-02-03 09:56:22,2018-02-03 10:33:41,2018-02-06 16:18:28,2018-02-16 17:28:48,2018-03-09 00:00:00,13.0
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06,0,de6dff97e5f1ba84a3cd9a3bc97df5f6,delivered,2017-04-09 17:41:13,2017-04-09 17:55:19,2017-04-10 14:24:47,2017-04-20 09:08:35,2017-05-10 00:00:00,10.0
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53,0,5986b333ca0d44534a156a52a8e33a83,delivered,2018-02-10 10:59:03,2018-02-10 15:48:21,2018-02-15 19:36:14,2018-02-28 16:33:35,2018-03-09 00:00:00,18.0


In [ ]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Train Logistic Regression
clf = LogisticRegression()
clf.fit(X_train, y_train)

# Predict
y_pred = clf.predict(X_test)

# Calculate metrics
print("\n--- Classification Performance Report ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.4f}")

# Output Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(f"TN: {cm[0,0]} | FP: {cm[0,1]}")
print(f"FN: {cm[1,0]} | TP: {cm[1,1]}")


--- Classification Performance Report ---
Accuracy:  0.8812
Precision: 0.7220
Recall:    0.1185
F1-Score:  0.2036

Confusion Matrix:
TN: 25034 | FP: 169
FN: 3266 | TP: 439


### 🔍 Interpreting the Classification Demo
Notice how the model achieves quite high accuracy, yet **Recall is incredibly low**.

Because low review scores are relatively rare (imbalanced), the Logistic Regression model plays it extremely safe. It rarely predicts a low score unless the delivery time was egregiously delayed. This illustrates why looking at Accuracy alone would deceive you into thinking the model is performing perfectly, when in fact it fails to flag the majority of unhappy customers!

---

## Section 6: Regression Metrics — MAE, RMSE and R²

When predicting a numeric variable, such as the delivery duration in days, we cannot use classification metrics like accuracy or precision. Instead, we measure the distance between the model's prediction ($\hat{y}_i$) and the actual ground truth ($y_i$).

Let's define the three most important metrics in regression:

### 1. Mean Absolute Error (MAE)
MAE is the average of the absolute differences between predictions and actual values.

$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

* **Intuition:** "On average, our predictions are off by $X$ units."
* **Pro:** Highly interpretable because it is in the exact same unit as our target variable (e.g., days).
* **Con:** Treats all errors equally. An error of 10 days is treated exactly 10 times worse than an error of 1 day.

### 2. Root Mean Squared Error (RMSE)
RMSE squares the errors before averaging them, then takes the square root.

$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$

* **Intuition:** Because errors are squared *before* they are averaged, **large errors are penalized heavily**.
* **When to prioritize:** When you want to avoid occasional large errors at all costs.

### 3. Coefficient of Determination ($R^2$ Score)
$R^2$ measures the proportion of variance in the target variable that is predictable from our feature variables.

$$R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

* **Intuition:** $R^2$ answers the question: *"How much better is our model compared to a baseline model that simply predicts the mean value of the dataset?"*
* **Values:**
  * $1.0$: Perfect predictions.
  * $0.0$: Performs exactly as badly as predicting the average value for every item.
  * Negative: Worse than predicting the mean.

---

## Demonstration B: Implementing a Regression Model on Olist

Let's build a model to predict the **actual delivery time (in days)** of Olist orders using features like freight value.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. We will merge orders with order items to get freight value
items_df = order_items.groupby('order_id')['freight_value'].sum().reset_index()
reg_data = pd.merge(orders_df, items_df, on='order_id').dropna(subset=['delivery_time_days', 'freight_value'])



In [ ]:
# Filter out extreme outliers or bad entries where delivery days < 0
reg_data = reg_data[reg_data['delivery_time_days'] >= 0]

X_reg = reg_data[['freight_value']]
y_reg = reg_data['delivery_time_days']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)

# Train Linear Regression
reg_model = LinearRegression()
reg_model.fit(X_train_r, y_train_r)



LinearRegression()

In [ ]:
# Predict
y_pred_r = reg_model.predict(X_test_r)

# Evaluate
mae = mean_absolute_error(y_test_r, y_pred_r)
mse = mean_squared_error(y_test_r, y_pred_r)
rmse = np.sqrt(mse)
r2 = reg_model.score(X_test_r, y_test_r)

print("--- Regression Performance Report ---")
print(f"Mean Absolute Error (MAE): {mae:.2f} Days")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} Days")
print(f"R-squared (R2): {r2:.4f}")

--- Regression Performance Report ---
Mean Absolute Error (MAE): 6.33 Days
Root Mean Squared Error (RMSE): 9.83 Days
R-squared (R2): 0.0271


### 🔍 Interpreting the Regression Demo
* **MAE is around {your value} days**: This means that on average, our delivery predictions are off by that many days.
* **RMSE is higher than MAE**: Why? Because of squaring. If the model makes a few massive mistakes (e.g. predicting a package will arrive in 10 days, but it actually takes 50 days), RMSE will penalize this heavily, dragging the score up.

---

## Demonstration C: Loss Function vs. Evaluation Metric

It is common to confuse **Loss Functions** and **Evaluation Metrics** because both measure error, but they serve different phases of machine learning:

1. **Loss Function:**
   - Used **during training**.
   - Must be **differentiable** so optimization algorithms (like gradient descent) can compute gradients to adjust weights.
   - Example: *Log Loss* (binary cross-entropy) or *Mean Squared Error*.
2. **Evaluation Metric:**
   - Used **after training** to assess performance.
   - Does not need to be differentiable. It must be **interpretable by humans** (e.g. F1-Score or accuracy).

Let's calculate MSE Loss manually on a tiny array of values to see how squaring penalises outliers.

In [ ]:
# Manual calculation showing how squaring penalises larger errors
actual_vals = np.array([10, 12, 11, 15])
pred_vals   = np.array([11, 12, 10, 25])  # An outlier error of 10 units on the last entry

absolute_errors = np.abs(actual_vals - pred_vals)
squared_errors = (actual_vals - pred_vals) ** 2

print("Absolute Errors:", absolute_errors)
print("Squared Errors: ", squared_errors)
print(f"\nManual MAE:  {np.mean(absolute_errors):.2f}")
print(f"Manual RMSE: {np.sqrt(np.mean(squared_errors)):.2f}")

Absolute Errors: [ 1  0  1 10]
Squared Errors:  [  1   0   1 100]

Manual MAE:  3.00
Manual RMSE: 5.05


---

## Section 7: Choosing the Right Metric — Decision Guide

Use this cheat sheet to choose metrics based on your business objectives:

| Metric | Task Type | Key Question It Answers | When to Use |
| :--- | :--- | :--- | :--- |
| **Accuracy** | Classification | What percentage of total predictions were correct? | When classes are balanced and all errors have equal costs. |
| **Precision** | Classification | Of all predicted positives, how many were correct? | When you want to minimize false positives (e.g., spam detection). |
| **Recall** | Classification | Of all actual positives, how many did we catch? | When you want to minimize false negatives (e.g., catching fraud). |
| **F1-Score** | Classification | How balanced is our model across precision & recall? | When you have imbalanced classes. |
| **MAE** | Regression | What is the average magnitude of error? | When you want an intuitive, robust error metric in original units. |
| **RMSE** | Regression | What is the penalized average error? | When occasional large errors are extremely undesirable. |
| **$R^2$** | Regression | How much variance does our model explain? | When comparing different models against a baseline mean predictor. |

---

## Section 8: Cross-Validation — Can We Trust This Score?

If we evaluate our model on a single train-test split, we might get lucky (or unlucky) with how the data was split. To ensure our model generalizes, we use **K-Fold Cross-Validation**.

### How K-Fold Cross-Validation Works:
1. Split the training data into $K$ equal-sized folds.
2. Train the model on $K-1$ folds and evaluate on the remaining $1$ fold.
3. Repeat this process $K$ times, using a different fold as the validation set each time.
4. Average the performance metrics across all $K$ runs to get a stable, reliable score.

In [ ]:
from sklearn.model_selection import cross_val_score

# Let's perform 5-fold cross-validation on our regression model
scores = cross_val_score(reg_model, X_reg, y_reg, cv=5, scoring='neg_mean_absolute_error')

# Scikit-learn cross-validation returns negative scores for optimization reasons
mae_scores = -scores

print("MAE across the 5 folds:", [round(x, 2) for x in mae_scores])
print(f"Mean Cross-Validation MAE: {mae_scores.mean():.2f} days (Standard Deviation: {mae_scores.std():.2f})")

MAE across the 5 folds: [np.float64(6.25), np.float64(6.19), np.float64(6.33), np.float64(6.3), np.float64(6.3)]
Mean Cross-Validation MAE: 6.27 days (Standard Deviation: 0.05)


---

## Section 9: Overfitting vs. Underfitting

Understanding model performance requires diagnosing generalization problems:

* **Underfitting:** The model is too simple to capture patterns in the data.
  * *Symptom:* High error on BOTH training and validation datasets.
* **Overfitting:** The model is overly complex and memorizes noise in the training data rather than generalizing.
  * *Symptom:* Extremely low error on training data, but high error on validation/test data.

## Demonstration D: Visualizing Overfitting with a Complex Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Model 1: Shallow Decision Tree (Underfit / Well-regularized)
model_simple = DecisionTreeRegressor(max_depth=3, random_state=42)
model_simple.fit(X_train_r, y_train_r)

# Model 2: Extremely Deep Decision Tree (Overfit)
model_complex = DecisionTreeRegressor(max_depth=50, random_state=42)
model_complex.fit(X_train_r, y_train_r)

# Evaluate both models on Training vs. Test Sets
print("--- Model Complexity Comparison (MAE in Days) ---")
print(f"Shallow Tree -> Train MAE: {mean_absolute_error(y_train_r, model_simple.predict(X_train_r)):.2f} | Test MAE: {mean_absolute_error(y_test_r, model_simple.predict(X_test_r)):.2f}")
print(f"Deep Tree    -> Train MAE: {mean_absolute_error(y_train_r, model_complex.predict(X_train_r)):.2f} | Test MAE: {mean_absolute_error(y_test_r, model_complex.predict(X_test_r)):.2f}")

--- Model Complexity Comparison (MAE in Days) ---
Shallow Tree -> Train MAE: 5.80 | Test MAE: 5.89
Deep Tree    -> Train MAE: 5.14 | Test MAE: 6.01


---

## Section 10: Worked Evaluation Case Study

Imagine Olist evaluates two customer retention models designed to flag orders at high risk of receiving bad reviews. Here are their scores on an identical validation dataset:

* **Model A:** Precision = $0.85$, Recall = $0.35$, F1-score = $0.50$
* **Model B:** Precision = $0.45$, Recall = $0.90$, F1-score = $0.60$

### Analysis:
* If the cost of calling an unhappy customer is very low (e.g. sending a automated push discount code), we should choose **Model B** because its high Recall ($90\%$) ensures we catch nearly all unhappy customers.
* If contacting customer service is highly resource-intensive and expensive (e.g. expensive live support outreach), we should choose **Model A** because its high Precision ($85\%$) ensures we do not waste costly support hours calling happy customers.

---

## Section 11: Assignment 1

**Q1:** A model that classifies transactional fraud has $99.9\%$ accuracy. Why might you still reject this model?

**Q2:** If a model has a precision of $1.0$, does it mean it makes zero errors? Explain.

**Q3:** Explain the difference between MAE and RMSE in your own words.


---

## Section 12: Assignment — Choose and Justify the Metric

**Task:** Read the following 3 scenario descriptions carefully. For each, determine the type of machine learning task (Classification or Regression), name the primary metric you would optimize for, and write 2-3 sentences justifying your choice based on the cost of prediction errors.

### Scenario 1:
Olist wants to build a model to flag transactions likely to be fraud. If a transaction is flagged as fraud, the credit card is immediately suspended, and the transaction is canceled.

### Scenario 2:
Olist wants to predict the estimated time of arrival (ETA) for high-value freight items. Giving a delivery date that is way too early is highly frustrating to customers and results in severe service penalties.

### Scenario 3:
Olist wants to automatically detect and group items into specific product categories based on description keywords.